In [ ]:
import os

from utils import setup_notebook

if not setup_notebook(
    required_keys=["LITELLM_API_KEY", "LITELLM_BASE_URL"],
):
    raise ValueError("Failed to setup notebook, please check your .env file")

masked_key_id = os.getenv("LITELLM_API_KEY")[:5] + "*" * (len(os.getenv("LITELLM_API_KEY")) - 5)
base_url = os.getenv("LITELLM_BASE_URL")
print(f"Using LiteLLM API Key: {masked_key_id}")
print(f"Using LiteLLM Base URL: {base_url}")

In [ ]:
from agent_platform.core.platforms.litellm import LiteLLMPlatformParameters
from agent_platform.core.platforms.llms_metadata_loader import llms_metadata_loader

llms_metadata_loader.load_data()

# Platform Parameters
parameters = LiteLLMPlatformParameters(
    litellm_api_key=os.getenv("LITELLM_API_KEY"), litellm_base_url=os.getenv("LITELLM_BASE_URL")
)

In [ ]:
from agent_platform.core.platforms.litellm.client import LiteLLMClient

litellm_client = LiteLLMClient(
    parameters=parameters,
)

In [ ]:
from typing import Annotated

from agent_platform.core.tools import ToolDefinition


async def joke_judge(joke: Annotated[str, "A joke to judge"]) -> bool:
    """Judge if a joke is funny.

    Arguments:
        joke: A joke to judge.

    Returns:
        True if the joke is funny, False otherwise.
    """
    return False


joke_judge_tool = ToolDefinition.from_callable(joke_judge)

print(joke_judge_tool)


async def random_number_generator(
    min_value: Annotated[int, "The minimum value of the range"],
    max_value: Annotated[int, "The maximum value of the range"],
) -> int:
    """Generate a random number.

    Arguments:
        min_value: The minimum value of the range.
        max_value: The maximum value of the range.

    Returns:
        A random number between min_value and max_value.
    """
    import random

    return random.randint(min_value, max_value)


random_number_generator_tool = ToolDefinition.from_callable(random_number_generator)

print(random_number_generator_tool)

In [ ]:
from agent_platform.core.prompts import PromptTextContent, PromptUserMessage
from agent_platform.core.prompts.prompt import Prompt

user_prompt = PromptUserMessage(
    content=[
        PromptTextContent(
            text="Please use the joke_judge tool to "
            "judge if the following joke is funny: "
            '"Why did the chicken cross the road?"\n'
            '"To get to the other side!"\n'
            "Also, generate a random number between 1 and 100.\n"
            "Run these tools in parallel.",
        ),
    ],
)
general_prompt = Prompt(
    system_instruction="You are a helpful assistant.",
    messages=[user_prompt],
    tools=[joke_judge_tool, random_number_generator_tool],
)

await general_prompt.finalize_messages()
litellm_prompt = await litellm_client.converters.convert_prompt(
    general_prompt,
    "litellm/openai/gpt-5",
)

print(litellm_prompt)

In [ ]:
deltas = []
i = 0
async for delta in litellm_client.generate_stream_response(
    litellm_prompt,
    "litellm/openai/gpt-5",
):
    print(f"CHUNK {i}: {delta!r}")
    deltas.append(delta)
    i += 1

In [ ]:
from pprint import pprint

from agent_platform.core.delta.combine_delta import combine_generic_deltas

assembled_dict = combine_generic_deltas(deltas)
pprint(assembled_dict)

In [ ]:
from agent_platform.core.responses import ResponseMessage

response_message = ResponseMessage.model_validate(assembled_dict)
pprint(response_message)

In [ ]:
from agent_platform.core.responses import ResponseToolUseContent

tool_use_content_block_joke_judge = response_message.content[1]
assert isinstance(tool_use_content_block_joke_judge, ResponseToolUseContent)

is_funny = await joke_judge(tool_use_content_block_joke_judge.tool_input["joke"])

print(f"Is the joke funny? {'Yes' if is_funny else 'No'}")

tool_use_content_block_rng = response_message.content[2]
assert isinstance(tool_use_content_block_rng, ResponseToolUseContent)

random_number = await random_number_generator(
    tool_use_content_block_rng.tool_input["min_value"],
    tool_use_content_block_rng.tool_input["max_value"],
)

print(f"Generated Random Number: {random_number}")

In [ ]:
from agent_platform.core.prompts import PromptTextContent, PromptToolResultContent

assert isinstance(tool_use_content_block_joke_judge, ResponseToolUseContent)
assert isinstance(tool_use_content_block_rng, ResponseToolUseContent)

tool_result_content_joke_judge = PromptToolResultContent(
    tool_name=tool_use_content_block_joke_judge.tool_name,
    tool_call_id=tool_use_content_block_joke_judge.tool_call_id,
    content=[PromptTextContent(text=str(is_funny))],
)

tool_result_content_rng = PromptToolResultContent(
    tool_name=tool_use_content_block_rng.tool_name,
    tool_call_id=tool_use_content_block_rng.tool_call_id,
    content=[PromptTextContent(text=str(random_number))],
)

print(tool_result_content_joke_judge)
print(tool_result_content_rng)

In [ ]:
from agent_platform.core.prompts import PromptToolUseContent
from agent_platform.core.thread import ThreadToolUsageContent

original_tool_use_joke_judge = response_message.content[1]
original_tool_use_rng = response_message.content[2]
assert isinstance(original_tool_use_joke_judge, ResponseToolUseContent)
assert isinstance(original_tool_use_rng, ResponseToolUseContent)

# Convert to ThreadToolUsageContent and then to PromptToolUseContent
ai_tool_use_joke_judge = ThreadToolUsageContent.from_response_tool_use(
    original_tool_use_joke_judge,
)
prompt_tool_use_joke_judge = PromptToolUseContent.from_thread_tool_usage(
    ai_tool_use_joke_judge,
)

ai_tool_use_rng = ThreadToolUsageContent.from_response_tool_use(original_tool_use_rng)
prompt_tool_use_rng = PromptToolUseContent.from_thread_tool_usage(ai_tool_use_rng)

pprint(prompt_tool_use_joke_judge)
pprint(prompt_tool_use_rng)

In [ ]:
from agent_platform.core.prompts import PromptAgentMessage

return_user_prompt = PromptUserMessage(
    content=[tool_result_content_joke_judge, tool_result_content_rng],
)
return_tool_use = PromptAgentMessage(
    content=[prompt_tool_use_joke_judge, prompt_tool_use_rng],
)
return_gen_prompt = Prompt(
    system_instruction="You are a helpful assistant.",
    messages=[user_prompt, return_tool_use, return_user_prompt],
    tools=[joke_judge_tool, random_number_generator_tool],
)
await return_gen_prompt.finalize_messages()
return_openai_prompt = await litellm_client.converters.convert_prompt(
    return_gen_prompt,
    model_id="litellm/openai/gpt-5",
)

pprint(return_openai_prompt)

In [ ]:
response = await litellm_client.generate_response(
    return_openai_prompt,
    "litellm/openai/gpt-5",
)

pprint(response)